In [1]:
import sys
import numpy as np
import pandas as pd
import scipy.stats as stats

from statsmodels.base.model import GenericLikelihoodModel
from IPython.display import display

# Change this if vuong_tests12.py is stored elsewhere.
sys.path.append("../")
import vuong_tests12 as vt


class JointNormal1(GenericLikelihoodModel):
    def loglikeobs(self, params):
        data = np.column_stack((self.endog, self.exog))
        return stats.multivariate_normal(
            mean=[params[0], 0.0],
            cov=[[1, 0], [0, 1]]
        ).logpdf(data)


class JointNormal2(GenericLikelihoodModel):
    def loglikeobs(self, params):
        data = np.column_stack((self.endog, self.exog))
        return stats.multivariate_normal(
            mean=[0.0, params[0]],
            cov=[[1, 0], [0, 1]]
        ).logpdf(data)


def setup_shi(yn, xn):
    n = len(yn)

    params1 = np.array([yn.mean()])
    model1 = JointNormal1(yn, xn)
    ll1 = model1.loglikeobs(params1)
    grad1 = model1.score_obs(params1).reshape(n, 1)
    hess1 = model1.hessian(params1)

    params2 = np.array([xn.mean()])
    model2 = JointNormal2(yn, xn)
    ll2 = model2.loglikeobs(params2)
    grad2 = model2.score_obs(params2).reshape(n, 1)
    hess2 = model2.hessian(params2)

    return ll1, grad1, hess1, params1, ll2, grad2, hess2, params2


def gen_data(rng, beta=0.0, nobs=200):
    # Your paper's Example 1 DGP.
    # beta=0 gives the equivalence null used in your size table.
    data = rng.multivariate_normal(
        mean=[beta, beta],
        cov=[[25, 0], [0, 1]],
        size=nobs
    )
    return data[:, 0], data[:, 1]

In [2]:
def compare_one_sample(args, rng, trials=1000, epsilon=.5,
                       alpha=.05, biascorrect=True):
    ll1, grad1, hess1, params1, ll2, grad2, hess2, params2 = args

    # Your observed statistic.
    N, omega, V, n = vt.sw_test_stat(*args, epsilon=epsilon)
    assert n % 2 == 0

    C = V.sum() / 2.0 if biascorrect else 0.0
    observed = (N + C) / (omega * np.sqrt(n))

    # Correct conditional center for singleton resampling.
    center = (1.0 + epsilon / 2.0) * (ll1 - ll2).sum()

    old_roots = np.empty(trials)
    corrected_roots = np.empty(trials)

    for b in range(trials):
        # Same bootstrap draws for both singleton versions.
        idx = rng.integers(0, n, size=n)

        N_star, omega_star, _ = vt.sw_test_stat(
            ll1[idx], grad1, hess1, params1,
            ll2[idx], grad2, hess2, params2,
            epsilon=epsilon,
            compute_v=False
        )

        denominator = omega_star * np.sqrt(n)

        # Your original singleton implementation.
        old_roots[b] = (N_star - N - C) / denominator

        # Correct conditional numerator center.
        # Bias-correction convention matches your pairwise routine.
        corrected_roots[b] = (N_star - center) / denominator

    # Your pairwise implementation, unchanged.
    pair_roots, pair_observed = vt.pairwise_bootstrap_distr(
        *args,
        epsilon=epsilon,
        trials=trials,
        biascorrect=biascorrect,
        seed=int(rng.integers(0, 2**31 - 1))
    )
    assert np.isclose(observed, pair_observed)

    def rejects(roots):
        if not np.all(np.isfinite(roots)):
            raise FloatingPointError("Nonfinite bootstrap statistic.")
        lo, hi = np.quantile(roots, [alpha / 2, 1 - alpha / 2])
        return observed < lo or observed > hi

    return [
        rejects(old_roots),
        rejects(corrected_roots),
        rejects(pair_roots)
    ]


def run_check(sample_sizes=(200,), num_sims=1000, trials=1000,
              epsilon=.5, alpha=.05, biascorrect=True, seed=12345):
    # Separate streams for original datasets and bootstrap draws.
    data_seed, boot_seed = np.random.SeedSequence(seed).spawn(2)
    data_rng = np.random.default_rng(data_seed)
    boot_rng = np.random.default_rng(boot_seed)

    names = [
        "Singleton: original",
        "Singleton: corrected",
        "Pairwise"
    ]
    rows = []

    for n in sample_sizes:
        rejected = np.empty((num_sims, 3), dtype=bool)

        for r in range(num_sims):
            # Generate data and fit your two models.
            yn, xn = gen_data(data_rng, beta=0.0, nobs=n)
            args = setup_shi(yn, xn)

            rejected[r] = compare_one_sample(
                args, boot_rng,
                trials=trials,
                epsilon=epsilon,
                alpha=alpha,
                biascorrect=biascorrect
            )

        rates = rejected.mean(axis=0)

        for name, rate in zip(names, rates):
            rows.append({
                "n": n,
                "method": name,
                "rejection_rate": rate,
                "MC_SE": np.sqrt(rate * (1 - rate) / num_sims)
            })

        print(f"Finished n={n}")

    return pd.DataFrame(rows)

In [3]:
results = run_check(
    sample_sizes=[200],       # Then expand to [100, 200, 500, 1000].
    num_sims=1000,
    trials=1000,
    epsilon=.5,
    alpha=.05,
    biascorrect=True,
    seed=12345
)

display(results.round(4))

Finished n=200


,n,method,rejection_rate,MC_SE
0,200,Singleton: original,0.343,0.0150
1,200,Singleton: corrected,0.041,0.0063
2,200,Pairwise,0.051,0.0070
